In [0]:
import json
from pyspark.sql.functions import *

In [0]:
# Use Spark and absolute path to bring in JSON content
spark_json_path = "/Workspace/Users/davesmith1095@gmail.com/data-5035-2026/week03/negotiated_rates.json"

# Read JSON file into a DataFrame (fix: add multiLine option)
df = spark.read.option('mode', 'PERMISSIVE').option('multiLine', True).json(spark_json_path)

# Check top few records of DataFrame
display(df)


In [0]:
# Use explode on lists similar to steps in Python
explode1 = df.withColumn("oon", explode("out_of_network"))

# Iteratively add new columns with exploded data
explode2 = explode1.withColumn("aa", explode("oon.allowed_amounts"))
explode3 = explode2.withColumn("pr", explode("aa.payments.providers"))

# Select final columns (can use dot notation to "traverse" dictionaries, e.g., allowed_amount)
finaldf = explode3.select(
    col("oon.name"),
    col("oon.billing_code_type"),
    col("oon.billing_code"),
    col("oon.description"),
    col("aa.service_code"),
    col("aa.billing_class"),
    col("aa.payments.allowed_amount"),
    col("pr.billed_charge"),
    col("pr.npi")[0].alias("npi")
)

display(finaldf)

### Recap
I started by observing the javascript, particularly the order and quantity of the layers. It came out like a Mortal Kombat cheatcode: DLDLDDDLDL. Dictionaries and Lists. This helped me visualize the schema and plan my approach for converting to a structured data frame. 

I initially tried the exercise in (mostly) Python, using a little Spark. I wrote out steps in plain English and prompted Gemini to come up with some starter code. I copied this over to think through the process, then retyped it to make comments, corrections, and adjustments. Using this initial approach to loop through lists/dictionaries and create an end dataframe was challenging but made sense.

I shifted to Spark from Python because the workflow made more sense after an initial run, but also because I realized I couldn't copy/paste JSON into my script and likely would use Spark in other big data scenarios. At first it followed a similar process, treating lists one way and dictionaries another--I liked the use of dot notation for dictionaries rather than string indexing. Reading in the JSON file with Spark actually circumvented an error I ran into earlier with the file formatting. The exploding and compilation of data was much simpler using all Spark instead of a mix of Spark and Python.

Spark was much faster than my initial run with Python. I thought at first I'd prefer Python for a similar task in the future, but I should probably just practice a bit more with Spark.